In [0]:
"""

# =========================================================
# PROJECT 2 - BANKING TRANSACTION PIPELINE
# =========================================================


DATASETS:
1. transactions.csv
2. customers.json
3. branches.parquet


1. Read and validate all datasets.
2. Convert transaction_date to DateType.
3. Filter failed transactions.
4. Identify duplicate transaction IDs.
5. Calculate total transaction amount per customer.
6. Find suspicious transactions greater than 1 lakh.
7. Join customers with transactions.
8. Find branches with highest transaction volume.
9. Calculate daily transaction totals.
10. Customers count based on the account type.
11. Use window functions to rank top customers.
12. Save final fraud analysis report.


# =========================================================
# PROJECT 3 - Application and Servers
# =========================================================


DATASETS:
1. application_logs.json
2. server_details.csv


1. Read log files.
2. Extract log_date and log_hour.
3. Filter ERROR logs.
4. Count errors by server.
5. Find most frequent error message.
6. Join logs with server details.
7. Calculate hourly error trend.
8. Save error summary report.

# =========================================================
# PROJECT 4 - HEALTHCARE DATA PIPELINE
# =========================================================

DATASETS:
1. patients.csv
2. appointments.json
3. doctors.parquet

1. Read and validate all datasets.
2. Convert appointment_date to DateType.
3. Filter cancelled appointments.
4. Count appointments by status.
5. Find total consultation fees per patient.
6. Find doctors with highest number of appointments.
7. Join patients with appointments.
8. Join appointments with doctors.
9. Find patients with no appointments (ANTI JOIN).
10. Calculate daily appointment counts.
11. Rank doctors by total consultation revenue.
12. Find the most common specialization.
13. Save final healthcare report.

"""

'\n\n# =========================================================\n# PROJECT 2 - BANKING TRANSACTION PIPELINE\n# =========================================================\n\n\nDATASETS:\n1. transactions.csv\n2. customers.json\n3. branches.parquet\n\n\n1. Read and validate all datasets.\n2. Convert transaction_date to DateType.\n3. Filter failed transactions.\n4. Identify duplicate transaction IDs.\n5. Calculate total transaction amount per customer.\n6. Find suspicious transactions greater than 1 lakh.\n7. Join customers with transactions.\n8. Find branches with highest transaction volume.\n9. Calculate daily transaction totals.\n10. Customers count based on the account type.\n11. Use window functions to rank top customers.\n12. Save final fraud analysis report.\n\n\n# =========================================================\n# PROJECT 3 - Application and Servers\n# =========================================================\n\n\nDATASETS:\n1. application_logs.json\n2. server_details.csv

In [0]:
# PROJECT 2 - BANKING TRANSACTION PIPELINE
# 1. Read and validate all datasets.

df_transactions = spark.table("dataframeassigment.etl2.transactions")

df_customers = spark.table("dataframeassigment.etl2.customers")

df_branches = spark.table("dataframeassigment.etl2.branches")

In [0]:
# PROJECT 2 - BANKING TRANSACTION PIPELINE
# 1. Read and validate all datasets.

df_transactions.printSchema()
df_customers.printSchema()
df_branches.printSchema()

print(df_transactions.count())
print(df_customers.count())
print(df_branches.count())


root
 |-- transaction_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- branch_id: long (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- amount: double (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- status: string (nullable = true)

root
 |-- account_type: string (nullable = true)
 |-- city: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- email: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)

root
 |-- branch_id: long (nullable = true)
 |-- branch_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- manager_name: string (nullable = true)

100
100
100


In [0]:
# PROJECT 2 - BANKING TRANSACTION PIPELINE
# 2. Convert transaction_date to DateType.

from pyspark.sql.functions import to_date, col
df_transactions.withColumn("transaction_date", to_date(col("transaction_date"), "yyyy-MM-dd")).printSchema()

root
 |-- transaction_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- branch_id: long (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- amount: double (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- status: string (nullable = true)



In [0]:
df_transactions.display()

transaction_id,customer_id,branch_id,transaction_date,amount,transaction_type,status
40,31,2,2023-05-21,4060.26,transfer,canceled
27,15,9,2020-05-14,5562.38,transfer,completed
2,15,2,2025-12-16,9060.33,deposit,pending
12,39,8,2024-04-09,842.51,transfer,failed
38,4,8,2021-03-19,6018.33,refund,failed
19,30,10,2023-08-27,161.5,withdrawal,failed
20,18,9,2021-10-25,7868.48,refund,canceled
5,25,7,2023-05-16,312.03,withdrawal,pending
40,43,6,2021-06-18,6390.31,transfer,failed
8,36,7,2023-11-07,4422.48,transfer,failed


In [0]:
df_customers.display()

account_type,city,customer_id,email,first_name,last_name
Free,Yaroslavskaya,26,rpidduck0@ustream.tv,Rudolfo,Pidduck
Enterprise,Dayu,13,adisdel1@shop-pro.jp,Alvinia,Disdel
Enterprise,Concepción,41,ltrever2@samsung.com,Livvy,Trever
Standard,Chaeryŏng-ŭp,28,tstennings3@wikimedia.org,Toma,Stennings
Free,Changtan,36,bcowser4@mlb.com,Beverlie,Cowser
Standard,Dibba Al-Hisn,40,bpalumbo5@usgs.gov,Brander,Palumbo
Premium,Petrich,22,dmcteague6@list-manage.com,Dayle,McTeague
Free,Kuala Lumpur,32,gewin7@about.com,Gilligan,Ewin
Standard,Surgut,43,mposvner8@e-recht24.de,Millard,Posvner
Enterprise,Ugbokpo,16,rkeyse9@de.vu,Randall,Keyse


In [0]:
# PROJECT 2 - BANKING TRANSACTION PIPELINE
# 3. Filter failed transactions.

from pyspark.sql.functions import col

df_transactions.filter(col("status")=="failed").display()


transaction_id,customer_id,branch_id,transaction_date,amount,transaction_type,status
12,39,8,2024-04-09,842.51,transfer,failed
38,4,8,2021-03-19,6018.33,refund,failed
19,30,10,2023-08-27,161.5,withdrawal,failed
40,43,6,2021-06-18,6390.31,transfer,failed
8,36,7,2023-11-07,4422.48,transfer,failed
3,5,9,2020-11-04,5860.03,purchase,failed
11,32,5,2020-08-07,6183.84,refund,failed
35,22,9,2023-03-19,5595.86,transfer,failed
20,45,7,2024-06-02,9090.95,refund,failed
13,22,4,2023-06-11,7697.0,deposit,failed


In [0]:
# PROJECT 2 - BANKING TRANSACTION PIPELINE
# 4. Identify duplicate transaction IDs.
from pyspark.sql.functions import col
df_transactions.groupBy("transaction_id").count().filter(col("count") > 1).orderBy(col("count").desc()).display()

transaction_id,count
20,7
14,6
38,5
40,5
5,5
8,4
24,4
11,4
39,4
12,4


In [0]:
# PROJECT 2 - BANKING TRANSACTION PIPELINE
# 5. Calculate total transaction amount per customer.
df_transactions.groupBy("customer_id").sum("amount").withColumnRenamed("sum(amount)", "total_amount").orderBy("total_amount").display()


customer_id,total_amount
25,312.03
29,758.49
42,1985.78
28,3417.44
36,4703.639999999999
30,5050.81
13,5469.33
20,5517.46
9,5866.31
37,7830.52


In [0]:
# PROJECT 2 - BANKING TRANSACTION PIPELINE
# 6. Find suspicious transactions greater than 1 lakh.
df_suspicious = df_transactions.filter(col("amount") > 100000)
df_suspicious.display()

transaction_id,customer_id,branch_id,transaction_date,amount,transaction_type,status


In [0]:
# PROJECT 2 - BANKING TRANSACTION PIPELINE
# 7. Join customers with transactions.
df_customer_transaction = df_customers.join(df_transactions, "customer_id", "inner")
df_customer_transaction.display()

customer_id,account_type,city,email,first_name,last_name,transaction_id,branch_id,transaction_date,amount,transaction_type,status
13,Enterprise,Dayu,adisdel1@shop-pro.jp,Alvinia,Disdel,14,7,2022-05-18,5469.33,fee,reversed
41,Enterprise,Concepción,ltrever2@samsung.com,Livvy,Trever,11,4,2024-05-13,5846.91,transfer,completed
28,Standard,Chaeryŏng-ŭp,tstennings3@wikimedia.org,Toma,Stennings,23,2,2026-08-29,3417.44,refund,completed
36,Free,Changtan,bcowser4@mlb.com,Beverlie,Cowser,9,9,2026-08-15,281.16,transfer,completed
40,Standard,Dibba Al-Hisn,bpalumbo5@usgs.gov,Brander,Palumbo,29,9,2026-02-19,8137.31,deposit,pending
22,Premium,Petrich,dmcteague6@list-manage.com,Dayle,McTeague,40,2,2021-07-15,9207.21,refund,failed
32,Free,Kuala Lumpur,gewin7@about.com,Gilligan,Ewin,12,10,2021-05-12,9762.4,fee,pending
43,Standard,Surgut,mposvner8@e-recht24.de,Millard,Posvner,14,3,2023-05-08,3905.8,fee,completed
16,Enterprise,Ugbokpo,rkeyse9@de.vu,Randall,Keyse,34,5,2021-08-26,7227.8,withdrawal,pending
20,Free,Şuwayr,lbenallacka@senate.gov,Lemmy,Benallack,40,6,2020-03-13,4253.28,fee,completed


In [0]:
# PROJECT 2 - BANKING TRANSACTION PIPELINE
# 8. Find branches with highest transaction volume.
df_highest_transaction = df_branches.join(df_transactions, "branch_id", "inner")
# df_highest_transaction.display()
df_highest_transaction.groupBy("branch_id").sum("amount").withColumnRenamed("sum(amount)", "total_amount").orderBy(col("total_amount").desc()).display()

branch_id,total_amount
6,964725.2999999998
9,792966.9000000001
5,707120.6399999999
3,702799.1999999998
2,477875.87999999995
7,468355.52
10,468309.18000000005
4,418332.86000000004
8,211817.10000000003
1,93336.36


In [0]:
# PROJECT 2 - BANKING TRANSACTION PIPELINE
# 9. Calculate daily transaction totals.
df_transactions.groupBy("transaction_date").sum("amount").alias("daily_transaction_totals").orderBy("transaction_date").display()

transaction_date,sum(amount)
2020-01-18,2805.73
2020-02-01,9779.12
2020-03-13,4253.28
2020-03-21,4829.73
2020-04-19,3048.34
2020-05-14,5562.38
2020-05-30,7284.28
2020-06-05,7124.7
2020-06-15,4978.38
2020-06-26,4137.74


In [0]:
# PROJECT 2 - BANKING TRANSACTION PIPELINE
# 10. Customers counts based on account type
df_customers.groupBy("account_type").count().display()

account_type,count
Free,25
Enterprise,29
Standard,20
Premium,26


In [0]:
# PROJECT 2 - BANKING TRANSACTION PIPELINE
# 11. Use window functions to rank top customers.

from pyspark.sql.functions import rank, col
from pyspark.sql.window import Window

df_customer_spending = df_customer_transaction.groupBy("customer_id").sum("amount").withColumnRenamed("sum(amount)", "total_amount")

window_spec = Window.orderBy(col("total_amount").desc())

df_ranked = df_customer_spending.withColumn("rank", rank().over(window_spec))
                                                                
df_ranked.display()



/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


customer_id,total_amount,rank
32,159938.69999999998,1
11,149928.80000000002,2
18,104672.28,3
17,81544.26,4
8,49280.67,5
19,45525.649999999994,6
22,45000.14,7
43,44593.04,8
15,42641.28,9
38,38626.08,10


In [0]:
# PROJECT 2 - BANKING TRANSACTION PIPELINE
# 12. Save final fraud analysis report.
df_fraud_report = df_transactions.groupBy("transaction_type").sum("amount").withColumnRenamed("sum(amount)", "total_amount").filter(col("total_amount") > 80000)
df_fraud_report.display()

transaction_type,total_amount
transfer,117975.08000000002
refund,99587.15000000002
fee,106541.91


In [0]:
# PROJECT 2 - BANKING TRANSACTION PIPELINE
# 12. Save final fraud analysis report.
df_fraud_report.write.mode("overwrite").saveAsTable("dataframeassigment.etl2.fraud_report")

In [0]:
# PROJECT 3 - Application and Servers
# 1. Read log files.
df_application_logs = spark.read.table("dataframeassigment.etl2.application_logs")
df_server_logs = spark.read.table("dataframeassigment.etl2.server_logs")

In [0]:
# PROJECT 3 - Application and Servers
# validate the records
df_application_logs.printSchema()
df_server_logs.printSchema()

print(df_application_logs.count())
print(df_server_logs.count())

root
 |-- endpoint: string (nullable = true)
 |-- log_id: long (nullable = true)
 |-- log_level: string (nullable = true)
 |-- log_timestamp: string (nullable = true)
 |-- message: string (nullable = true)
 |-- response_time_ms: long (nullable = true)
 |-- server_id: long (nullable = true)
 |-- status_code: long (nullable = true)

root
 |-- server_id: long (nullable = true)
 |-- server_name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- environment: string (nullable = true)
 |-- owner_team: string (nullable = true)

100
100


In [0]:
# PROJECT 3 - Application and Servers
# 2. Extract log_date and log_hour.

from pyspark.sql.functions import to_timestamp, to_date, hour, col

df_application_logs = df_application_logs.withColumn("log_timestamp", to_timestamp(col("log_timestamp"), "M/d/yyyy"))
df_application_logs = df_application_logs.withColumn("log_date", to_date(col("log_timestamp")))
df_application_logs = df_application_logs.withColumn("log_hour", hour(col("log_timestamp")))

df_application_logs.display()



endpoint,log_id,log_level,log_timestamp,message,response_time_ms,server_id,status_code,log_date,log_hour
/profile,1,debug,2024-12-23T00:00:00.000Z,"Nam congue, risus semper porta volutpat, quam pede lobortis ligula, sit amet eleifend pede libero quis orci. Nullam molestie nibh in lectus. Pellentesque at nulla. Suspendisse potenti.",2031,1,503,2024-12-23,0
/login,2,error,2026-05-11T00:00:00.000Z,"Aliquam non mauris. Morbi non lectus. Aliquam sit amet diam in magna bibendum imperdiet. Nullam orci pede, venenatis non, sodales sed, tincidunt eu, felis. Fusce posuere felis sed lacus.",7446,10,503,2026-05-11,0
/logout,3,debug,2025-06-11T00:00:00.000Z,"Cum sociis natoque penatibus et magnis dis parturient montes, nascetur ridiculus mus. Etiam vel augue. Vestibulum rutrum rutrum neque. Aenean auctor gravida sem. Praesent id massa id nisl venenatis lacinia. Aenean sit amet justo. Morbi ut odio. Cras mi pede, malesuada in, imperdiet et, commodo vulputate, justo. In blandit ultrices enim. Lorem ipsum dolor sit amet, consectetuer adipiscing elit.",346,5,400,2025-06-11,0
/payment,4,warn,2023-11-26T00:00:00.000Z,"Proin leo odio, porttitor id, consequat in, consequat ut, nulla. Sed accumsan felis. Ut at dolor quis odio consequat varius. Integer ac leo.",9539,1,200,2023-11-26,0
/profile,5,debug,2024-03-19T00:00:00.000Z,"Nulla ac enim. In tempor, turpis nec euismod scelerisque, quam turpis adipiscing lorem, vitae mattis nibh ligula nec sem. Duis aliquam convallis nunc. Proin at turpis a pede posuere nonummy. Integer non velit. Donec diam neque, vestibulum eget, vulputate ut, ultrices vel, augue. Vestibulum ante ipsum primis in faucibus orci luctus et ultrices posuere cubilia Curae; Donec pharetra, magna vestibulum aliquet ultrices, erat tortor sollicitudin mi, sit amet lobortis sapien sapien non mi.",9294,10,503,2024-03-19,0
/login,6,warn,2023-11-05T00:00:00.000Z,"Fusce congue, diam id ornare imperdiet, sapien urna pretium nisl, ut volutpat sapien arcu sed augue.",4830,6,404,2023-11-05,0
/login,7,info,2023-03-29T00:00:00.000Z,"Integer aliquet, massa id lobortis convallis, tortor risus dapibus augue, vel accumsan tellus nisi eu orci. Mauris lacinia sapien quis libero. Nullam sit amet turpis elementum ligula vehicula consequat. Morbi a ipsum. Integer a nibh. In quis justo. Maecenas rhoncus aliquam lacus. Morbi quis tortor id nulla ultrices aliquet.",308,2,200,2023-03-29,0
/profile,8,info,2024-12-12T00:00:00.000Z,"Morbi vestibulum, velit id pretium iaculis, diam erat fermentum justo, nec condimentum neque sapien placerat ante. Nulla justo. Aliquam quis turpis eget elit sodales scelerisque.",7347,2,201,2024-12-12,0
/products,9,error,2023-01-24T00:00:00.000Z,Praesent blandit lacinia erat. Vestibulum sed magna at nunc commodo placerat.,5439,1,404,2023-01-24,0
/logout,10,info,2023-05-09T00:00:00.000Z,"Nulla tellus. In sagittis dui vel nisl. Duis ac nibh. Fusce lacus purus, aliquet at, feugiat non, pretium quis, lectus.",27,2,404,2023-05-09,0


In [0]:
# PROJECT 3 - Application and Servers
# 3. Filter ERROR logs.
df_application_logs.filter(col("log_level")=="error").display()

endpoint,log_id,log_level,log_timestamp,message,response_time_ms,server_id,status_code,log_date,log_hour
/login,2,error,2026-05-11T00:00:00.000Z,"Aliquam non mauris. Morbi non lectus. Aliquam sit amet diam in magna bibendum imperdiet. Nullam orci pede, venenatis non, sodales sed, tincidunt eu, felis. Fusce posuere felis sed lacus.",7446,10,503,2026-05-11,0
/products,9,error,2023-01-24T00:00:00.000Z,Praesent blandit lacinia erat. Vestibulum sed magna at nunc commodo placerat.,5439,1,404,2023-01-24,0
/logout,11,error,2024-03-21T00:00:00.000Z,"In eleifend quam a odio. In hac habitasse platea dictumst. Maecenas ut massa quis augue luctus tincidunt. Nulla mollis molestie lorem. Quisque ut erat. Curabitur gravida nisi at nibh. In hac habitasse platea dictumst. Aliquam augue quam, sollicitudin vitae, consectetuer eget, rutrum at, lorem.",884,1,400,2024-03-21,0
/logout,18,error,2025-04-12T00:00:00.000Z,Vestibulum ante ipsum primis in faucibus orci luctus et ultrices posuere cubilia Curae; Duis faucibus accumsan odio. Curabitur convallis. Duis consequat dui nec nisi volutpat eleifend. Donec ut dolor. Morbi vel lectus in quam fringilla rhoncus.,702,7,200,2025-04-12,0
/logout,22,error,2026-04-24T00:00:00.000Z,"Nulla tempus. Vivamus in felis eu sapien cursus vestibulum. Proin eu mi. Nulla ac enim. In tempor, turpis nec euismod scelerisque, quam turpis adipiscing lorem, vitae mattis nibh ligula nec sem. Duis aliquam convallis nunc.",6981,9,404,2026-04-24,0
/profile,26,error,2025-09-19T00:00:00.000Z,"Quisque erat eros, viverra eget, congue eget, semper rutrum, nulla. Nunc purus. Phasellus in felis. Donec semper sapien a libero. Nam dui. Proin leo odio, porttitor id, consequat in, consequat ut, nulla.",9218,4,503,2025-09-19,0
/login,28,error,2025-10-11T00:00:00.000Z,Vestibulum rutrum rutrum neque. Aenean auctor gravida sem. Praesent id massa id nisl venenatis lacinia. Aenean sit amet justo.,2938,1,500,2025-10-11,0
/profile,31,error,2023-11-13T00:00:00.000Z,Aenean sit amet justo.,8616,6,404,2023-11-13,0
/logout,39,error,2025-01-19T00:00:00.000Z,Ut at dolor quis odio consequat varius. Integer ac leo. Pellentesque ultrices mattis odio.,1090,4,503,2025-01-19,0
/payment,45,error,2025-04-23T00:00:00.000Z,"Morbi sem mauris, laoreet ut, rhoncus aliquet, pulvinar sed, nisl. Nunc rhoncus dui vel sem. Sed sagittis. Nam congue, risus semper porta volutpat, quam pede lobortis ligula, sit amet eleifend pede libero quis orci. Nullam molestie nibh in lectus. Pellentesque at nulla. Suspendisse potenti. Cras in purus eu magna vulputate luctus. Cum sociis natoque penatibus et magnis dis parturient montes, nascetur ridiculus mus.",9539,6,400,2025-04-23,0


In [0]:
# PROJECT 3 - Application and Servers
# 4. Count errors by server.

df_Counterror_by_server = df_application_logs.filter(col("log_level")=="error").groupBy("server_id").count()
df_Counterror_by_server.display()

server_id,count
10,1
1,3
7,2
9,5
4,4
6,3
2,1
3,1
5,1


In [0]:
# PROJECT 3 - Application and Servers
# 5. Find most frequent error message.
df_application_logs.filter(col("log_level")=="error").groupBy("message").count().orderBy(col("count").desc()).display()


message,count
"Nunc rhoncus dui vel sem. Sed sagittis. Nam congue, risus semper porta volutpat, quam pede lobortis ligula, sit amet eleifend pede libero quis orci.",1
"Nulla facilisi. Cras non velit nec nisi vulputate nonummy. Maecenas tincidunt lacus at velit. Vivamus vel nulla eget eros elementum pellentesque. Quisque porta volutpat erat. Quisque erat eros, viverra eget, congue eget, semper rutrum, nulla. Nunc purus. Phasellus in felis. Donec semper sapien a libero.",1
"Pellentesque at nulla. Suspendisse potenti. Cras in purus eu magna vulputate luctus. Cum sociis natoque penatibus et magnis dis parturient montes, nascetur ridiculus mus. Vivamus vestibulum sagittis sapien.",1
Nullam porttitor lacus at turpis. Donec posuere metus vitae ipsum. Aliquam non mauris. Morbi non lectus. Aliquam sit amet diam in magna bibendum imperdiet.,1
"Integer aliquet, massa id lobortis convallis, tortor risus dapibus augue, vel accumsan tellus nisi eu orci. Mauris lacinia sapien quis libero. Nullam sit amet turpis elementum ligula vehicula consequat. Morbi a ipsum. Integer a nibh. In quis justo. Maecenas rhoncus aliquam lacus. Morbi quis tortor id nulla ultrices aliquet. Maecenas leo odio, condimentum id, luctus nec, molestie sed, justo.",1
"Donec quis orci eget orci vehicula condimentum. Curabitur in libero ut massa volutpat convallis. Morbi odio odio, elementum eu, interdum eu, tincidunt in, leo. Maecenas pulvinar lobortis est. Phasellus sit amet erat. Nulla tempus.",1
"Morbi sem mauris, laoreet ut, rhoncus aliquet, pulvinar sed, nisl. Nunc rhoncus dui vel sem. Sed sagittis. Nam congue, risus semper porta volutpat, quam pede lobortis ligula, sit amet eleifend pede libero quis orci. Nullam molestie nibh in lectus. Pellentesque at nulla. Suspendisse potenti. Cras in purus eu magna vulputate luctus. Cum sociis natoque penatibus et magnis dis parturient montes, nascetur ridiculus mus.",1
"Nunc purus. Phasellus in felis. Donec semper sapien a libero. Nam dui. Proin leo odio, porttitor id, consequat in, consequat ut, nulla. Sed accumsan felis. Ut at dolor quis odio consequat varius.",1
"Vestibulum ante ipsum primis in faucibus orci luctus et ultrices posuere cubilia Curae; Duis faucibus accumsan odio. Curabitur convallis. Duis consequat dui nec nisi volutpat eleifend. Donec ut dolor. Morbi vel lectus in quam fringilla rhoncus. Mauris enim leo, rhoncus sed, vestibulum sit amet, cursus id, turpis. Integer aliquet, massa id lobortis convallis, tortor risus dapibus augue, vel accumsan tellus nisi eu orci. Mauris lacinia sapien quis libero. Nullam sit amet turpis elementum ligula vehicula consequat. Morbi a ipsum.",1
Aenean sit amet justo.,1


In [0]:
# PROJECT 3 - Application and Servers
# 6. Join logs with server details.

df_application_logs.join(df_server_logs, "server_id", "inner").display()

server_id,endpoint,log_id,log_level,log_timestamp,message,response_time_ms,status_code,log_date,log_hour,server_name,region,environment,owner_team
1,/profile,1,debug,2024-12-23T00:00:00.000Z,"Nam congue, risus semper porta volutpat, quam pede lobortis ligula, sit amet eleifend pede libero quis orci. Nullam molestie nibh in lectus. Pellentesque at nulla. Suspendisse potenti.",2031,503,2024-12-23,0,Photospace,Europe,QA,DevOps
10,/login,2,error,2026-05-11T00:00:00.000Z,"Aliquam non mauris. Morbi non lectus. Aliquam sit amet diam in magna bibendum imperdiet. Nullam orci pede, venenatis non, sodales sed, tincidunt eu, felis. Fusce posuere felis sed lacus.",7446,503,2026-05-11,0,Rooxo,India,QA,Backend
5,/logout,3,debug,2025-06-11T00:00:00.000Z,"Cum sociis natoque penatibus et magnis dis parturient montes, nascetur ridiculus mus. Etiam vel augue. Vestibulum rutrum rutrum neque. Aenean auctor gravida sem. Praesent id massa id nisl venenatis lacinia. Aenean sit amet justo. Morbi ut odio. Cras mi pede, malesuada in, imperdiet et, commodo vulputate, justo. In blandit ultrices enim. Lorem ipsum dolor sit amet, consectetuer adipiscing elit.",346,400,2025-06-11,0,Tazz,Singapore,QA,Platform
1,/payment,4,warn,2023-11-26T00:00:00.000Z,"Proin leo odio, porttitor id, consequat in, consequat ut, nulla. Sed accumsan felis. Ut at dolor quis odio consequat varius. Integer ac leo.",9539,200,2023-11-26,0,Photospace,Europe,QA,DevOps
10,/profile,5,debug,2024-03-19T00:00:00.000Z,"Nulla ac enim. In tempor, turpis nec euismod scelerisque, quam turpis adipiscing lorem, vitae mattis nibh ligula nec sem. Duis aliquam convallis nunc. Proin at turpis a pede posuere nonummy. Integer non velit. Donec diam neque, vestibulum eget, vulputate ut, ultrices vel, augue. Vestibulum ante ipsum primis in faucibus orci luctus et ultrices posuere cubilia Curae; Donec pharetra, magna vestibulum aliquet ultrices, erat tortor sollicitudin mi, sit amet lobortis sapien sapien non mi.",9294,503,2024-03-19,0,Rooxo,India,QA,Backend
6,/login,6,warn,2023-11-05T00:00:00.000Z,"Fusce congue, diam id ornare imperdiet, sapien urna pretium nisl, ut volutpat sapien arcu sed augue.",4830,404,2023-11-05,0,Avamm,India,QA,Frontend
2,/login,7,info,2023-03-29T00:00:00.000Z,"Integer aliquet, massa id lobortis convallis, tortor risus dapibus augue, vel accumsan tellus nisi eu orci. Mauris lacinia sapien quis libero. Nullam sit amet turpis elementum ligula vehicula consequat. Morbi a ipsum. Integer a nibh. In quis justo. Maecenas rhoncus aliquam lacus. Morbi quis tortor id nulla ultrices aliquet.",308,200,2023-03-29,0,Yombu,India,Production,Backend
2,/profile,8,info,2024-12-12T00:00:00.000Z,"Morbi vestibulum, velit id pretium iaculis, diam erat fermentum justo, nec condimentum neque sapien placerat ante. Nulla justo. Aliquam quis turpis eget elit sodales scelerisque.",7347,201,2024-12-12,0,Yombu,India,Production,Backend
1,/products,9,error,2023-01-24T00:00:00.000Z,Praesent blandit lacinia erat. Vestibulum sed magna at nunc commodo placerat.,5439,404,2023-01-24,0,Photospace,Europe,QA,DevOps
2,/logout,10,info,2023-05-09T00:00:00.000Z,"Nulla tellus. In sagittis dui vel nisl. Duis ac nibh. Fusce lacus purus, aliquet at, feugiat non, pretium quis, lectus.",27,404,2023-05-09,0,Yombu,India,Production,Backend


In [0]:
# PROJECT 3 - Application and Servers
# 7. Calculate hourly error trend.
df_application_logs.filter(col("log_level")=="error").groupBy("log_hour").count().display()

log_hour,count
0,21


In [0]:
# PROJECT 3 - Application and Servers
# 8. Save error summary report.
df_error_summary = df_application_logs.groupBy("server_id").count().withColumnRenamed("count","error_count")
df_error_summary.display()

server_id,error_count
1,14
10,10
5,9
6,12
2,17
8,3
7,9
9,12
3,6
4,8


In [0]:
df_error_summary.write.mode("overwrite").saveAsTable("dataframeassigment.etl2.error_summary")

In [0]:
# PROJECT 4 - HEALTHCARE DATA PIPELINE
# 1. Read and validate all datasets.

df_patients = spark.read.csv("/Volumes/dataframeassigment/etl3/extrack_transfer_load/patients.csv",header=True, inferSchema=True)


df_appointments = spark.read.json("/Volumes/dataframeassigment/etl3/extrack_transfer_load/appointments.json")

df_doctors = spark.read.csv("/Volumes/dataframeassigment/etl3/extrack_transfer_load/doctors.csv",header=True, inferSchema=True)

In [0]:
# PROJECT 4 - HEALTHCARE DATA PIPELINE
# 1. Read and validate all datasets.

df_patients.printSchema()
df_appointments.printSchema()
df_doctors.printSchema()

print(df_patients.count())
print(df_appointments.count())
print(df_doctors.count())

root
 |-- patient_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- blood_group: string (nullable = true)
 |-- is_active: boolean (nullable = true)

root
 |-- _corrupt_record: string (nullable = true)
 |-- appointment_date: string (nullable = true)
 |-- appointment_id: long (nullable = true)
 |-- appointment_type: string (nullable = true)
 |-- consultation_fee: double (nullable = true)
 |-- doctor_id: long (nullable = true)
 |-- patient_id: long (nullable = true)
 |-- status: string (nullable = true)

root
 |-- doctor_id: integer (nullable = true)
 |-- doctor_name: string (nullable = true)
 |-- specialization: string (nullable = true)
 |-- hospital_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- experience_years: integer (nullable = true)

200
200
200


In [0]:
# PROJECT 4 - HEALTHCARE DATA PIPELINE
# 2. Convert appointment_date to DateType
from pyspark.sql.functions import to_date, col

df_appointments.withColumn("appointment_date", to_date(col("appointment_date"), "M/d/yyyy")).display()

_corrupt_record,appointment_date,appointment_id,appointment_type,consultation_fee,doctor_id,patient_id,status
null,2024-04-21,1,Dermatology,7268.42,19,34,Completed
null,2030-02-23,2,Ophthalmology,13624.61,10,27,Pending
null,2028-08-07,3,Dental,2879.71,18,32,Pending
null,2023-08-21,4,Cardiology,13073.7,12,29,Cancelled
null,2025-03-24,5,Pediatrics,2285.3,10,3,Cancelled
null,2021-08-08,6,Gynecology,8846.38,18,45,Completed
null,2030-12-16,7,ENT,13649.62,10,38,Completed
null,2025-05-13,8,Ophthalmology,11721.93,1,17,Pending
null,2024-02-26,9,Orthopedic,1892.33,11,15,Pending
null,2023-12-21,10,Ophthalmology,14875.93,6,40,Cancelled


In [0]:
# PROJECT 4 - HEALTHCARE DATA PIPELINE
# 3. Filter cancelled appointments.
df_appointments.filter(col("status")=="Cancelled").display()

_corrupt_record,appointment_date,appointment_id,appointment_type,consultation_fee,doctor_id,patient_id,status
null,8/21/2023,4,Cardiology,13073.7,12,29,Cancelled
null,3/24/2025,5,Pediatrics,2285.3,10,3,Cancelled
null,12/21/2023,10,Ophthalmology,14875.93,6,40,Cancelled
null,4/13/2030,11,Gynecology,13905.73,2,4,Cancelled
null,1/13/2023,17,Ophthalmology,10831.26,4,15,Cancelled
null,5/10/2026,19,Dental,14629.62,13,21,Cancelled
null,4/13/2024,20,Dermatology,14635.71,17,45,Cancelled
null,4/13/2022,21,General Checkup,7384.46,10,14,Cancelled
null,9/21/2021,26,Urology,1260.86,18,3,Cancelled
null,2/14/2022,28,Ophthalmology,3698.36,18,8,Cancelled


In [0]:
# PROJECT 4 - HEALTHCARE DATA PIPELINE
# 4. Count appointments by status.
df_appointments.groupBy("status").count().display()

status,count
Pending,66
Cancelled,78
Completed,56


In [0]:
# PROJECT 4 - HEALTHCARE DATA PIPELINE
# 5. Find total consultation fees per patient.
df_appointments.groupBy("doctor_id").sum("consultation_fee").alias("total_fee").display()


doctor_id,sum(consultation_fee)
11,133482.26
9,130142.04999999999
14,203120.79
17,99548.79999999999
3,34799.71
19,87450.11000000002
4,103757.84
6,92201.90000000001
20,90717.81
13,68119.79000000001


In [0]:
# PROJECT 4 - HEALTHCARE DATA PIPELINE
# 6. Find doctors with highest number of appointments.
df_doctor_appoinmnets = df_appointments.join(df_doctors, "doctor_id", "full")
df_doctor_appoinmnets.groupBy("doctor_id", "doctor_name", "experience_years").count().orderBy(col("count").desc()).display()

doctor_id,doctor_name,experience_years,count
10,Ailsun Beades,26,19
14,Neila Fesby,18,16
16,Lolita Norrington,30,16
18,Cy Maundrell,12,15
11,Melonie Feakins,3,14
5,Jarvis McCrory,22,12
9,Patsy Kentish,14,11
4,Viv Gigg,1,11
6,Clarissa Riddiford,18,11
1,Ali Filpi,2,10


In [0]:
# PROJECT 4 - HEALTHCARE DATA PIPELINE
# 7. Join patients with appointments.
df_patients.join(df_appointments, "patient_id", "inner").display()

patient_id,first_name,last_name,gender,age,city,blood_group,is_active,_corrupt_record,appointment_date,appointment_id,appointment_type,consultation_fee,doctor_id,status
34,Lee,Orrom,Agender,67,Cigedang,O+,false,null,4/21/2024,1,Dermatology,7268.42,19,Completed
27,Leoine,Phinnis,Female,69,Jomboy,O-,false,null,2/23/2030,2,Ophthalmology,13624.61,10,Pending
32,Isadora,Antonikov,Female,62,Matiompong,A+,false,null,8/7/2028,3,Dental,2879.71,18,Pending
29,Brande,Turmall,Female,54,Markog,B-,true,null,8/21/2023,4,Cardiology,13073.7,12,Cancelled
3,Robinette,Rexworthy,Female,72,Novosmolinskiy,O+,true,null,3/24/2025,5,Pediatrics,2285.3,10,Cancelled
45,Bruis,Brockman,Male,51,Hanjia,AB-,true,null,8/8/2021,6,Gynecology,8846.38,18,Completed
38,Foster,Cowdroy,Male,21,Yatsuomachi-higashikumisaka,A-,false,null,12/16/2030,7,ENT,13649.62,10,Completed
17,Clement,Malone,Male,50,Cedynia,O-,true,null,5/13/2025,8,Ophthalmology,11721.93,1,Pending
15,Elysee,Izaac,Female,68,Międzyzdroje,B-,false,null,2/26/2024,9,Orthopedic,1892.33,11,Pending
40,Pollyanna,Nicely,Female,42,Tashang,AB-,true,null,12/21/2023,10,Ophthalmology,14875.93,6,Cancelled


In [0]:
# PROJECT 4 - HEALTHCARE DATA PIPELINE
# 8. Join appointments with doctors.
df_appointments.join(df_doctors, "doctor_id", "inner").display()

doctor_id,_corrupt_record,appointment_date,appointment_id,appointment_type,consultation_fee,patient_id,status,doctor_name,specialization,hospital_name,city,experience_years
19,null,4/21/2024,1,Dermatology,7268.42,34,Completed,Cordy Poynor,Dermatology,Agimba,San Juan,22
10,null,2/23/2030,2,Ophthalmology,13624.61,27,Pending,Ailsun Beades,Cardiology,Browsebug,Dengmingsi,26
18,null,8/7/2028,3,Dental,2879.71,32,Pending,Cy Maundrell,Neurology,Reallinks,Dachong,12
12,null,8/21/2023,4,Cardiology,13073.7,29,Cancelled,Malchy Durtnel,Neurology,Dynabox,Évry,24
10,null,3/24/2025,5,Pediatrics,2285.3,3,Cancelled,Ailsun Beades,Cardiology,Browsebug,Dengmingsi,26
18,null,8/8/2021,6,Gynecology,8846.38,45,Completed,Cy Maundrell,Neurology,Reallinks,Dachong,12
10,null,12/16/2030,7,ENT,13649.62,38,Completed,Ailsun Beades,Cardiology,Browsebug,Dengmingsi,26
1,null,5/13/2025,8,Ophthalmology,11721.93,17,Pending,Ali Filpi,Neurology,Snaptags,Dagang,2
11,null,2/26/2024,9,Orthopedic,1892.33,15,Pending,Melonie Feakins,Neurology,Eamia,Cuauhtemoc,3
6,null,12/21/2023,10,Ophthalmology,14875.93,40,Cancelled,Clarissa Riddiford,Neurology,Demizz,Sundbyberg,18


In [0]:
# PROJECT 4 - HEALTHCARE DATA PIPELINE
# 9. Find patients with no appointments (ANTI JOIN).
df_patients.join(df_appointments, "patient_id", "left_anti").display()

patient_id,first_name,last_name,gender,age,city,blood_group,is_active
9,Adorne,Reinard,Female,70,Tanzhou,A+,true
12,Vinny,Ervin,Female,60,Jaguarari,AB-,true
44,Maxy,Batstone,Genderfluid,60,Stroitel’,A-,true
48,Tonya,Yakushkev,Female,35,Mochumí,A+,true
51,Othella,McMenamin,Bigender,48,Hongmen,O+,true
52,Sherry,Tomczynski,Female,37,Sidi Bouzid,A+,false
53,Sioux,Pepye,Genderqueer,33,Harstad,AB+,true
54,Nedda,Apple,Female,41,Gunungangka,B-,false
55,Johnathon,MacGaughey,Male,60,Halton,O+,true
56,Ilaire,Bantham,Male,30,Orvault,A-,true


In [0]:
# PROJECT 4 - HEALTHCARE DATA PIPELINE
# 10. Calculate daily appointment counts.
df_appointments.groupBy("appointment_date").count().display()

appointment_date,count
8/7/2028,1
8/1/2022,1
11/29/2021,1
10/5/2023,1
2/20/2029,1
4/28/2030,1
8/31/2028,1
6/5/2021,1
4/9/2020,1
7/5/2028,1


In [0]:
# PROJECT 4 - HEALTHCARE DATA PIPELINE
# 11. Rank doctors by total consultation revenue.

from pyspark.sql.functions import rank, col
from pyspark.sql.window import Window

df_revenue = df_appointments.groupBy("doctor_id").sum("consultation_fee").withColumnRenamed("sum(consultation_fee)", "total_fee")
window_spec = Window.orderBy(col("total_fee").desc())
df_revenue.withColumn("rank", rank().over(window_spec)).display()


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


doctor_id,total_fee,rank
14,203120.79,1
10,197057.34999999998,2
5,155540.44,3
16,138620.39,4
18,138311.02000000002,5
11,133482.26,6
9,130142.04999999999,7
1,112973.05,8
4,103757.84,9
17,99548.79999999999,10


In [0]:
# PROJECT 4 - HEALTHCARE DATA PIPELINE
# 12. Find the most common specialization.
df_appointments.join(df_doctors, "doctor_id", "inner").groupBy("specialization").count().orderBy(col("count").desc()).display()

specialization,count
Neurology,78
Cardiology,38
Dermatology,34
Pediatrics,16
Oncology,15
Psychiatry,10
Orthopedics,9


In [0]:
# PROJECT 4 - HEALTHCARE DATA PIPELINE
# 13. Save final healthcare report.
df_final_report = df_patients.join(df_appointments, "patient_id", "inner").join(df_doctors, "doctor_id", "inner")

In [0]:
df_final_report_clean = df_final_report.drop(df_patients.city)
df_final_report_clean.write.mode("overwrite").parquet("/Volumes/dataframeassigment/etl3/extrack_transfer_load/final_healthcare_report")